# TP Kaggle - Isidro Perasso
Desarrollo del modelo final, compuesto por un ensamble de tres modelos, para la mejor predicción personal lograda en la competencia de Kaggle AP_2026Q1 - TP3 de la Materia Análisis Predictivo ITBA

Para detalle de todas las notebooks de exploración de datos y prueba de modelos, dirigirse al siguiente link de GitHub: https://github.com/ichiP245/TP_Kaggle_Predictivo/tree/main

Importamos librerías e instalamos el módulo de *catboost*

In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import r2_score, mean_squared_error, classification_report
from sklearn.inspection import permutation_importance
import scipy.stats as stats
from catboost import CatBoostRegressor, Pool, CatBoostClassifier

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

## Carga de datos

Cargamos los datos de train y los que vamos a predecir de Kaggle

- Acá agarramos el ID de la base de train que subí a Google Drive, porque este tiene *feature engineering* y tratamiento de outliers previo que sería innecesario replicar acá y ya está en el GitHub

In [4]:
!pip install gdown
file_id = '1IoqDAcw5kLL1TisGMX_jGKFvXDjycqBm'
output_filename = 'df_train_2_FE.csv'

!gdown --id "{file_id}" -O "{output_filename}"

# Actualiza url_train para usar el archivo descargado localmente
url_train = output_filename

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1IoqDAcw5kLL1TisGMX_jGKFvXDjycqBm
To: /content/df_train_2_FE.csv
100% 42.4M/42.4M [00:00<00:00, 65.6MB/s]


Les hacemos algunas transformaciones y *feature engineering* que no tenían cargados en la última versión de la base

In [5]:
# Leemos el primer dataframe (train)
# url_train = '/content/gdrive/MyDrive/TP Kaggle - Isidro Perasso/df_train_2_FE.csv'
df = pd.read_csv(url_train)

# Leemos el segundo dataframe (Kaggle)
url_kaggle = 'https://raw.githubusercontent.com/ichiP245/TP_Kaggle_Predictivo/refs/heads/main/Datasets/df_val_2_FE_sinOutliers.csv'
df_val = pd.read_csv(url_kaggle)

# Sacamos nulos
df.dropna(inplace=True)

# Arreglamos algunas variables
df['key'] = df['key'].astype(str)
df['mode'] = df['mode'].astype(str)
df['grupo_anio'] = df['grupo_anio'].astype(str)
df['len_playlist_name'] = df['len_playlist_name'].astype(str)
df['tonality'] = df['tonality'].astype(str)
df['distance_mean_duration_min'] = (df['duration_min'] - df['duration_min'].mean())
df['distance_sq_mean_duration_min'] = (df['distance_mean_duration_min'])**2
df['log_duration_min'] = np.log(df['duration_min'])
df['z_duration_min'] = (df['duration_min'] - df['duration_min'].mean())/df['duration_min'].std()

# Hacemos Feature Engineering de tiempo (para train)
df['tiene_mes_y_dia'] = (df['track_album_release_date'].apply(lambda x: x.count('-')) == 2).astype(int)
serie_tiempo = pd.to_datetime(df.loc[df['tiene_mes_y_dia'] == 1, 'track_album_release_date'])
df['mes'] = 13
df.loc[df['tiene_mes_y_dia'] == 1, 'mes'] = serie_tiempo.dt.month
df['dia'] = 32
df.loc[df['tiene_mes_y_dia'] == 1, 'dia'] = serie_tiempo.dt.day
df['tiene_mes_y_dia'] = df['tiene_mes_y_dia'].astype(str)
df['mes'] = df['mes'].astype(str)
df['dia'] = df['dia'].astype(str)

# Repetimos lo de arriba en el df_val que es el de Kaggle
df_val['tiene_mes_y_dia'] = (df_val['track_album_release_date'].apply(lambda x: x.count('-')) == 2).astype(int)
serie_tiempo = pd.to_datetime(df_val.loc[df_val['tiene_mes_y_dia'] == 1, 'track_album_release_date'])
df_val['mes'] = 13
df_val.loc[df_val['tiene_mes_y_dia'] == 1, 'mes'] = serie_tiempo.dt.month
df_val['dia'] = 32
df_val.loc[df_val['tiene_mes_y_dia'] == 1, 'dia'] = serie_tiempo.dt.day
df_val['tiene_mes_y_dia'] = df_val['tiene_mes_y_dia'].astype(str)
df_val['mes'] = df_val['mes'].astype(str)
df_val['dia'] = df_val['dia'].astype(str)
df_val['log_duration_min'] = np.log(df_val['duration_min'])
df_val['z_duration_min'] = (df_val['duration_min'] - df['duration_min'].mean())/df['duration_min'].std()

## Modelo 1: CatBoost Regressor con Data Leakage

Creamos un dataframe reducido para entrenar el primer CatBoost Regressor. Como dice el título, es un CatBoost con variables con Data Leakage, ya que las variables de distancia de la media, para cada variable numérica, están calculadas respecto de la media del dataset de Kaggle de validación.

Esto se realizó considerando las grandes diferencias entre el dataset de entrenamiento y el de validación de Kaggle. De esta manera, el CatBoost aprende las distancias en relación a lo que va a tener en el dataset de Kaggle.

In [ ]:
# Creamos un nuevo df que es el que va a entrar al modelo
df_inicial = df[['Unnamed: 0', 'track_id', 'track_name', 'track_artist','track_popularity', 'track_album_id', 'track_album_name',
       'playlist_name', 'playlist_id','playlist_genre', 'playlist_subgenre', 'danceability', 'energy','loudness', 'speechiness',
        'acousticness', 'instrumentalness','liveness', 'valence', 'tempo', 'duration_min', 'grupo_anio', 'tonality',
        'tiene_mes_y_dia', 'mes', 'dia']]
# Calculamos variables de distancias con leakage
df_inicial_FE = df_inicial.copy()
df_inicial_FE['distance_mean_danceability'] = (df_inicial_FE['danceability'] - df_val['danceability'].mean())
df_inicial_FE['distance_sq_mean_danceability'] = (df_inicial_FE['distance_mean_danceability'])**2
df_inicial_FE['distance_mean_energy'] = (df_inicial_FE['energy'] - df_val['energy'].mean())
df_inicial_FE['distance_sq_mean_energy'] = (df_inicial_FE['distance_mean_energy'])**2
df_inicial_FE['distance_mean_loudness'] = (df_inicial_FE['loudness'] - df_val['loudness'].mean())
df_inicial_FE['distance_sq_mean_loudness'] = (df_inicial_FE['distance_mean_loudness'])**2
df_inicial_FE['distance_mean_speechiness'] = (df_inicial_FE['speechiness'] - df_val['speechiness'].mean())
df_inicial_FE['distance_sq_mean_speechiness'] = (df_inicial_FE['distance_mean_speechiness'])**2
df_inicial_FE['distance_mean_acousticness'] = (df_inicial_FE['acousticness'] - df_val['acousticness'].mean())
df_inicial_FE['distance_sq_mean_acousticness'] = (df_inicial_FE['distance_mean_acousticness'])**2
df_inicial_FE['distance_mean_liveness'] = (df_inicial_FE['liveness'] - df_val['liveness'].mean())
df_inicial_FE['distance_sq_mean_liveness'] = (df_inicial_FE['distance_mean_liveness'])**2
df_inicial_FE['distance_mean_valence'] = (df_inicial_FE['valence'] - df_val['valence'].mean())
df_inicial_FE['distance_sq_mean_valence'] = (df_inicial_FE['distance_mean_valence'])**2
df_inicial_FE['distance_mean_tempo'] = (df_inicial_FE['tempo'] - df_val['tempo'].mean())
df_inicial_FE['distance_sq_mean_tempo'] = (df_inicial_FE['distance_mean_tempo'])**2
df_inicial_FE['distance_mean_duration_min'] = (df_inicial_FE['duration_min'] - df_val['duration_min'].mean())
df_inicial_FE['distance_sq_mean_duration_min'] = (df_inicial_FE['distance_mean_duration_min'])**2

Separamos el dataset de entrenamiento entre variables independientes y variable target. Notese que quitamos varias variables relativas a la canción (track) y a la playlist buscando que el modelo generalice mejor y no memorice identificadores solamente.

In [ ]:
# Separamos en X e y
X = df_inicial_FE.drop(columns=['Unnamed: 0', 'track_popularity'])
y = df_inicial_FE['track_popularity']

# Sacamos algunas variables que podian hacer muy notable el overfitting
X_reduced = X.drop(columns=['track_name','track_album_name','playlist_name','playlist_id','playlist_genre','playlist_subgenre'])
X_train, X_test, y_train, y_test = train_test_split(X_reduced, y, test_size=0.2, random_state=42)
cat_features = X_reduced.select_dtypes(include='object').columns.tolist()
train_pool = Pool(X_train, y_train, cat_features=cat_features)

model_catboost_DL = CatBoostRegressor(
    iterations=1200,
    learning_rate=0.01,
    loss_function='RMSE',
    eval_metric='R2',
    random_seed=42,
    verbose=100
)

# ── ENTRENAMIENTO ──────────────────────────────────────────────
model_catboost_DL.fit(train_pool)

print('TRAIN')
y_pred = model_catboost_DL.predict(X_train)
print(f"R²:   {r2_score(y_train, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_pred)):.4f}")

print('TEST')
y_pred = model_catboost_DL.predict(X_test)
print(f"R²:   {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

0:	learn: 0.0055168	total: 895ms	remaining: 17m 53s
100:	learn: 0.2726930	total: 11.3s	remaining: 2m 2s
200:	learn: 0.3705629	total: 15.2s	remaining: 1m 15s
300:	learn: 0.3937823	total: 19.8s	remaining: 59s
400:	learn: 0.4048310	total: 23.8s	remaining: 47.5s
500:	learn: 0.4125231	total: 29.2s	remaining: 40.7s
600:	learn: 0.4181480	total: 33.7s	remaining: 33.6s
700:	learn: 0.4232485	total: 38.1s	remaining: 27.1s
800:	learn: 0.4278569	total: 44s	remaining: 21.9s
900:	learn: 0.4319411	total: 48.5s	remaining: 16.1s
1000:	learn: 0.4362931	total: 53.2s	remaining: 10.6s
1100:	learn: 0.4403404	total: 59s	remaining: 5.3s
1199:	learn: 0.4437758	total: 1m 3s	remaining: 0us
TRAIN
R²:   0.7622
RMSE: 12.1763
TEST
R²:   0.4878
RMSE: 17.9344


Habiendo corrido el modelo vemos un R² de 0.4878 en test y 0.7622 en train. La diferencia es de casi 30 puntos porcentuales pero entendemos que se puede deber a algo de overfitting necesario para aumentar nuestras predicciones en Kaggle también.

Pasamos a predecir, entonces hacemos el dataframe que vamos a predecir adaptado con las columnas que necesita el modelo de entrada

In [ ]:
# Hacemos el modelo de df_val que va a entrar al modelo para ser predicho
df_val_inicial = df_val[['Unnamed: 0', 'track_id', 'track_name', 'track_artist','track_album_id', 'track_album_name',
                         'playlist_name', 'playlist_id','playlist_genre', 'playlist_subgenre','danceability', 'energy',
                         'loudness', 'speechiness', 'acousticness', 'instrumentalness','liveness', 'valence', 'tempo',
                         'duration_min', 'grupo_anio', 'tonality','tiene_mes_y_dia', 'mes', 'dia']].copy()
df_val_inicial_FE = df_val_inicial.copy()
df_val_inicial_FE = df_val_inicial_FE.drop(columns=['Unnamed: 0', 'track_name','track_album_name','playlist_name','playlist_id','playlist_genre','playlist_subgenre'])
df_val_inicial_FE['distance_mean_danceability'] = (df_val_inicial_FE['danceability'] - df_val['danceability'].mean())
df_val_inicial_FE['distance_sq_mean_danceability'] = (df_val_inicial_FE['distance_mean_danceability'])**2
df_val_inicial_FE['distance_mean_energy'] = (df_val_inicial_FE['energy'] - df_val['energy'].mean())
df_val_inicial_FE['distance_sq_mean_energy'] = (df_val_inicial_FE['distance_mean_energy'])**2
df_val_inicial_FE['distance_mean_loudness'] = (df_val_inicial_FE['loudness'] - df_val['loudness'].mean())
df_val_inicial_FE['distance_sq_mean_loudness'] = (df_val_inicial_FE['distance_mean_loudness'])**2
df_val_inicial_FE['distance_mean_speechiness'] = (df_val_inicial_FE['speechiness'] - df_val['speechiness'].mean())
df_val_inicial_FE['distance_sq_mean_speechiness'] = (df_val_inicial_FE['distance_mean_speechiness'])**2
df_val_inicial_FE['distance_mean_acousticness'] = (df_val_inicial_FE['acousticness'] - df_val['acousticness'].mean())
df_val_inicial_FE['distance_sq_mean_acousticness'] = (df_val_inicial_FE['distance_mean_acousticness'])**2
df_val_inicial_FE['distance_mean_liveness'] = (df_val_inicial_FE['liveness'] - df_val['liveness'].mean())
df_val_inicial_FE['distance_sq_mean_liveness'] = (df_val_inicial_FE['distance_mean_liveness'])**2
df_val_inicial_FE['distance_mean_valence'] = (df_val_inicial_FE['valence'] - df_val['valence'].mean())
df_val_inicial_FE['distance_sq_mean_valence'] = (df_val_inicial_FE['distance_mean_valence'])**2
df_val_inicial_FE['distance_mean_tempo'] = (df_val_inicial_FE['tempo'] - df_val['tempo'].mean())
df_val_inicial_FE['distance_sq_mean_tempo'] = (df_val_inicial_FE['distance_mean_tempo'])**2
df_val_inicial_FE['distance_mean_duration_min'] = (df_val_inicial_FE['duration_min'] - df_val['duration_min'].mean())
df_val_inicial_FE['distance_sq_mean_duration_min'] = (df_val_inicial_FE['distance_mean_duration_min'])**2

# Predecimos
y_kaggle_catboost_DL = model_catboost_DL.predict(df_val_inicial_FE[X_reduced.columns])

## Modelo 2: CatBoost Regressor con imputación *track_id*

Creamos el dataframe con las columnas que va a usar este segundo CatBoost Regressor. Ahora sí se usan algunas variables de la playlist, pero las de track, no. Además, se crearon binarias para ver si una canción era de "r&b" o no, que es el único género que comparten las bases de train y Kaggle, con el objetivo de que pueda diferenciar ese género, que es el único con el que sí se va a encontrar en el dataset para predecir.

In [ ]:
cols = ['Unnamed: 0','track_artist', 'playlist_name', 'playlist_id',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'grupo_anio',
       'distance_mean_danceability', 'distance_sq_mean_danceability', 'distance_mean_energy',
       'distance_sq_mean_energy', 'distance_mean_loudness',
       'distance_sq_mean_loudness', 'distance_mean_speechiness',
       'distance_sq_mean_speechiness', 'distance_mean_acousticness',
       'distance_sq_mean_acousticness', 'distance_mean_liveness',
       'distance_sq_mean_liveness', 'distance_mean_valence',
       'distance_sq_mean_valence', 'distance_mean_tempo',
       'distance_sq_mean_tempo', 'dance_energy', 'happy_energy',
       'acoustic_instr', 'duration_min', 'track_age', 'len_playlist_name',
       'log_instrumentalness', 'log_speechiness', 'log_liveness', 'z_danceability', 'z_energy', 'z_loudness', 'z_tempo',
       'z_valence', 'z_acousticness', 'z_instrumentalness', 'z_speechiness',
       'global_atipicality', 'max_z_score',
       'most_extreme_feature', 'mood_score', 'chill_score', 'melancholy_score',
       'radio_friendly', 'club_score', 'acoustic_contrast', 'mood_complexity',
       'dance_val_tension', 'vocal_presence', 'pure_instrumental',
       'spoken_word', 'loudness_abs', 'loudness_energy_ratio',
       'dynamic_compression', 'live_vocal', 'studio_clean', 'danceability_sq',
       'danceability_sqrt', 'energy_sq', 'energy_sqrt', 'valence_sq',
       'valence_sqrt', 'loudness_abs_sq', 'loudness_abs_sqrt', 'tonality',
        'tiene_mes_y_dia', 'mes', 'dia']

dummies_genre = pd.get_dummies(df['playlist_genre']).astype(int)
dummies_subgenre = pd.get_dummies(df['playlist_subgenre']).astype(int)
genre_imp = dummies_genre[['r&b']]
subgenre_imp = dummies_subgenre[['neo soul']]

copia = df.copy()

df_catboost_2 = pd.concat([copia[cols], genre_imp, subgenre_imp], axis=1)

df_catboost_2['distance_mean_duration_min'] = (df['duration_min'] - df['duration_min'].mean())
df_catboost_2['distance_sq_mean_duration_min'] = (df_catboost_2['distance_mean_duration_min'])**2
df_catboost_2['log_duration_min'] = np.log(df['duration_min'])
df_catboost_2['z_duration_min'] = (df['duration_min'] - df['duration_min'].mean())/df['duration_min'].std()

df_catboost_2['r&b'] = df_catboost_2['r&b'].astype(str)
df_catboost_2['neo soul'] = df_catboost_2['neo soul'].astype(str)

Entrenamos el modelo. En esta ocasión, creamos un set de validación (que no es el de Kaggle, sino que nos referimos a un conjunto de los datos de train) que se van a usar para ir evaluando mientras entrena el modelo y poder frenar si está overfitteando. Se definieron 1000 iteraciones, un *learning rate* de 0.025 y 50 iteraciones para que haya *early stopping*.

In [ ]:
X = df_catboost_2.copy()
y = df['track_popularity'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_true, X_val, y_train_true, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

cat_features = df_catboost_2.select_dtypes(include='object').columns.tolist()
train_pool = Pool(X_train_true, y_train_true, cat_features=cat_features)
val_pool = Pool(X_val, y_val, cat_features=cat_features)
test_pool  = Pool(X_test,  y_test,  cat_features=cat_features)

model_catboost_2 = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.025,
    loss_function='RMSE',
    eval_metric='RMSE',
    early_stopping_rounds=50,
    random_seed=42,
    verbose=100
)

# ── ENTRENAMIENTO ──────────────────────────────────────────────
model_catboost_2.fit(train_pool, eval_set=val_pool)

print('TRAIN')
y_pred = model_catboost_2.predict(X_train_true)
print(f"R²:   {r2_score(y_train_true, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train_true, y_pred)):.4f}")

print('VAL')
y_pred = model_catboost_2.predict(X_val)
print(f"R²:   {r2_score(y_val, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_pred)):.4f}")

print('TEST')
y_pred = model_catboost_2.predict(X_test)
print(f"R²:   {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

0:	learn: 24.8557575	test: 24.5192869	best: 24.5192869 (0)	total: 73.3ms	remaining: 1m 13s
100:	learn: 19.6094958	test: 18.7816328	best: 18.7816328 (100)	total: 10.3s	remaining: 1m 31s
200:	learn: 18.9315285	test: 18.3463453	best: 18.3463453 (200)	total: 18.1s	remaining: 1m 12s
300:	learn: 18.6493962	test: 18.2006670	best: 18.2005905 (299)	total: 24.6s	remaining: 57.1s
400:	learn: 18.3691478	test: 18.0548437	best: 18.0545414 (399)	total: 32.3s	remaining: 48.3s
500:	learn: 18.1339993	test: 17.9618714	best: 17.9618714 (500)	total: 38.7s	remaining: 38.5s
600:	learn: 17.9085295	test: 17.9098818	best: 17.9098818 (600)	total: 46.4s	remaining: 30.8s
700:	learn: 17.7098554	test: 17.8665727	best: 17.8665727 (700)	total: 52.9s	remaining: 22.5s
800:	learn: 17.5373390	test: 17.8341091	best: 17.8341091 (800)	total: 1m 1s	remaining: 15.2s
900:	learn: 17.3760925	test: 17.8087393	best: 17.8067425 (898)	total: 1m 8s	remaining: 7.58s
999:	learn: 17.2260671	test: 17.7893951	best: 17.7893951 (999)	total: 

Vemos que el rendimiento en test fue mejor aún que en validación, por lo que vemos que aunque haya overfitting respecto al dataset de train (más de 20 puntos de R^2 de diferencia) el modelo logra predecir acorde al set de validación.

Adaptamos el dataframe de datos de Kaggle para que pueda entrar al modelo de CatBoost y predecimos.

In [ ]:
df_val_ML = df_val.copy()
df_val_ML = df_val_ML[cols]

df_val_ML['r&b'] = (df_val['playlist_genre'] == 'r&b').astype(int).rename('r&b')
df_val_ML['neo soul'] = (df_val['playlist_subgenre'] == 'neo soul').astype(int).rename('neo soul')
df_val_ML['distance_mean_duration_min'] = (df_val_ML['duration_min'] - df['duration_min'].mean())
df_val_ML['distance_sq_mean_duration_min'] = (df_val_ML['distance_mean_duration_min'])**2
df_val_ML['log_duration_min'] = np.log(df['duration_min'])
df_val_ML['z_duration_min'] = (df['duration_min'] - df['duration_min'].mean())/df['duration_min'].std()
df_val_ML['r&b'] = df_val_ML['r&b'].astype(str)
df_val_ML['neo soul'] = df_val_ML['neo soul'].astype(str)

y_kaggle_catboost_2 = model_catboost_2.predict(df_val_ML[df_catboost_2.columns])

Dado que hay una cantidad importante de valores de track_id que coinciden con los de Kaggle, vamos a emplear una metodología en la que si el track_id de la canción a predecir del dataset de Kaggle está en el dataset de train, entonces reemplazamos la predicción del CatBoost por el promedio de track_popularity de ese valor de track_id en train.

In [ ]:
# Calculamos el promedio de track_popularity para cada track_id del df de train
track_popularity_avg_df = df.groupby('track_id')['track_popularity'].mean()

# Creamos una serie con el ID de la fila como índice, en el que si encuentra el track_id de la canción
# entre las que también están en train, entonces toma ese valor; sino, deja vacío
y_val_predicted_track_id_avg = df_val.set_index('Unnamed: 0')['track_id'].map(track_popularity_avg_df)

# Crea una serie con las predicciones del modelo CatBoost indexadas por el ID de Kaggle de esa fila
y_val_pred_series = pd.Series(y_kaggle_catboost_2, index=df_val['Unnamed: 0'])

# Combinamos: si la serie tiene un valor promedio de track_id para ese dato, usa ese valor; sino, usa el valor predicho
y_val_combined_predictions = y_val_predicted_track_id_avg.fillna(y_val_pred_series)

# Creamos el dataframe final de predicciones de este modelo
submission_df_combined_track_id = pd.DataFrame({
    'ID': y_val_combined_predictions.index,
    'track_popularity_track_id': y_val_combined_predictions.values
})

## CatBoost Classifier 0s

Volvemos a cargar el dataset (igual que arriba)  porque puedieron haber unos cambios y el modelo es sensible a que esté todo igual para poder llegar al rendimiento conocido.

In [ ]:
df = pd.read_csv(url_train)
df_val = pd.read_csv(url_kaggle)

df.dropna(inplace=True)
df['key'] = df['key'].astype(str)
df['mode'] = df['mode'].astype(str)
df['grupo_anio'] = df['grupo_anio'].astype(str)
df['len_playlist_name'] = df['len_playlist_name'].astype(str)
df['tonality'] = df['tonality'].astype(str)

# FE de tiempo
df['tiene_mes_y_dia'] = (df['track_album_release_date'].apply(lambda x: x.count('-')) == 2).astype(int)
serie_tiempo = pd.to_datetime(df.loc[df['tiene_mes_y_dia'] == 1, 'track_album_release_date'])
df['mes'] = 13
df.loc[df['tiene_mes_y_dia'] == 1, 'mes'] = serie_tiempo.dt.month
df['dia'] = 32
df.loc[df['tiene_mes_y_dia'] == 1, 'dia'] = serie_tiempo.dt.day
df['tiene_mes_y_dia'] = df['tiene_mes_y_dia'].astype(str)
df['mes'] = df['mes'].astype(str)
df['dia'] = df['dia'].astype(str)

# Lo mismo para validation
df_val['tiene_mes_y_dia'] = (df_val['track_album_release_date'].apply(lambda x: x.count('-')) == 2).astype(int)
serie_tiempo = pd.to_datetime(df_val.loc[df_val['tiene_mes_y_dia'] == 1, 'track_album_release_date'])
df_val['mes'] = 13
df_val.loc[df_val['tiene_mes_y_dia'] == 1, 'mes'] = serie_tiempo.dt.month
df_val['dia'] = 32
df_val.loc[df_val['tiene_mes_y_dia'] == 1, 'dia'] = serie_tiempo.dt.day
df_val['tiene_mes_y_dia'] = df_val['tiene_mes_y_dia'].astype(str)
df_val['mes'] = df_val['mes'].astype(str)
df_val['dia'] = df_val['dia'].astype(str)

Separamos train, validación y test. Acá, la variable target no es continua (el track_popularity) sino binaria:

*   Si el track_popularity es 0, entonces la variable toma 1, y si track_popularity no es 0 entonces la variable  toma el valor 0.

De esta manera, queremos saber la probabilidad de que una observación tome el valor 0

Entrenamos el CatBoost Classifier con 1500 iteraciones, *learning rate* de 0.015 y 250 iteraciones para llegar al *early stopping*

In [ ]:
X = df.drop(columns=['es0','track_popularity', 'track_popularity_bins', 'anio', 'track_album_release_date',
                     'playlist_name', 'playlist_id', 'playlist_genre', 'playlist_subgenre', 'Unnamed: 0'])
y = df['es0']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

cat_features=X.select_dtypes(include='object').columns.to_list()
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_val, y_val, cat_features=cat_features)


catboost_c = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.015,
    loss_function='Logloss',
    eval_metric='Recall',
    random_seed=42,
    verbose=100,
    depth=5,
    early_stopping_rounds=250)

# ── ENTRENAMIENTO ──────────────────────────────────────────────
catboost_c.fit(train_pool, eval_set=val_pool)

print('\nTRAIN')
y_pred = catboost_c.predict(X_train)
print(classification_report(y_train, y_pred))

print('VAL')
y_pred = catboost_c.predict(X_val)
print(classification_report(y_val, y_pred))

print('TEST')
y_pred = catboost_c.predict(X_test)
print(classification_report(y_test, y_pred))

0:	learn: 0.0000000	test: 0.0000000	best: 0.0000000 (0)	total: 155ms	remaining: 3m 52s
100:	learn: 0.2234130	test: 0.3531353	best: 0.3531353 (95)	total: 7.29s	remaining: 1m 40s
200:	learn: 0.2300082	test: 0.3630363	best: 0.3630363 (195)	total: 14.2s	remaining: 1m 31s
300:	learn: 0.2374279	test: 0.3630363	best: 0.3630363 (195)	total: 21.4s	remaining: 1m 25s
400:	learn: 0.2456719	test: 0.3696370	best: 0.3696370 (398)	total: 29.8s	remaining: 1m 21s
500:	learn: 0.2530915	test: 0.3729373	best: 0.3729373 (416)	total: 36.8s	remaining: 1m 13s
600:	learn: 0.2555647	test: 0.3729373	best: 0.3762376 (504)	total: 45.3s	remaining: 1m 7s
700:	learn: 0.2662819	test: 0.3762376	best: 0.3762376 (504)	total: 53.4s	remaining: 1m
Stopped by overfitting detector  (250 iterations wait)

bestTest = 0.3762376238
bestIteration = 504

Shrink model to first 505 iterations.

TRAIN
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     15593
           1       0.99    

Lo que más nos importa son las métricas sobre el caso positivo, que es la minoría, ya que hay pocos valores en 0 respecto del total y el dataset está desbalanceado para esta tarea de clasificación. Vemos que en train el modelo overfittea, pero entre validación y test performa bastante parecido, con un leve deterioro en el recall y una mejora en la precision.

Predecimos

In [ ]:
prediccion = catboost_c.predict_proba(df_val[X_train.columns.to_list()])

## Predicciones Finales

Creamos un df con cada predicción. Para el caso del clasificador nos quedamos con la primera columna que representa la probabilidad de NO ser 0.

In [ ]:
submission_leakage = pd.DataFrame({'ID': df_val['Unnamed: 0'],
                                  'track_popularity': y_kaggle_catboost_DL})

submission_track_id = pd.DataFrame({'ID': df_val['Unnamed: 0'],
                                    'track_popularity': submission_df_combined_track_id['track_popularity_track_id']})

df_predicciones = pd.DataFrame({'ID': df_val['Unnamed: 0'],
                                'prob_ser0': prediccion[:,0]})

Unimos todas estas columnas en un mismo dataframe y hacemos lo siguiente:


*   Multiplicamos cada prediccion que obtuvimos de los modelos CatBoost Regressor (en el segundo caso con la imputación por track_id) por la probabilida que dio el CatBoost Classifier de que esa observación NO sea 0 -> logrando así que las filas con probabilidad más alta de NO ser 0, se mantengan en valores similares a los predichos; mientras que las filas con probabilidades más bajas de NO ser 0 (o sea probabilidas más alta de serlo), reducen su valor predicho, obteniendo como un castigo

* Estas predicciones las elevamos a la cuarta y al cubo, respectivamente, para que sean incluso más bajas, y porque obtuvimos los mejores resultados con esos valores de ese "hiperparámetro"  

* Luego, como la predicción de track_id siempre fue mejor que la de Data Leakage, ponderamos esa en un 75% y la otra en 25%

Fórmula: 0,25 * CatBoost_1 * ProbNoSerCero^4 + 0,75 * CatBoost_2 * ProbNoSerCero^3

In [ ]:
df_submission = pd.merge(submission_leakage, submission_track_id, on='ID', how='outer')
df_submission = pd.merge(df_submission, df_predicciones, on='ID', how='outer')

print(f'Quedaron {df_submission.isna().sum().sum()} valores nulos')

df_submission['leakage_ponderado^4'] = submission_leakage['track_popularity']*(df_predicciones['prob_ser0']**4)
df_submission['track_id_ponderado^3'] = submission_track_id['track_popularity']*(df_predicciones['prob_ser0']**3)

prediccion_kaggle = pd.DataFrame({
    'ID':df_val['Unnamed: 0'],
    'track_popularity': df_submission['leakage_ponderado^4']*0.25 + df_submission['track_id_ponderado^3']*0.75
})

Quedaron 0 valores nulos


In [ ]:
prediccion_kaggle.to_csv('MejorPrediccion.csv',index=False)